# 08 — Paired Robustness Benchmark untuk Artikel JRI

**Pertanyaan penelitian:** Dari tiga classifier yang memakai embedding Swin-Tiny yang sama—**Deep-FCN**, **Logistic Regression**, dan **LightGBM**—mana yang paling stabil ketika foto mata ikan mengalami perubahan pencahayaan atau blur?

Notebook ini **tidak melatih ulang** Swin-Tiny maupun classifier. Notebook membaca model yang sudah dihasilkan notebook 02–04, menguji **setiap citra test yang sama** pada kondisi asli dan enam gangguan terkontrol, kemudian menghasilkan tabel, grafik, evaluasi per kelas/per spesies, dan interval bootstrap berpasangan untuk draft artikel JRI.

**Cara menjalankan:** `Runtime → Change runtime type → T4 GPU` (disarankan), kemudian **Runtime → Run all**. Semua output dibuat di folder `04_results/metrics/robustness_jri` dan `04_results/figures/robustness_jri`, sehingga file eksperimen sebelumnya tidak ditimpa.

**Batas interpretasi:** pengujian gangguan sintetis ≠ validasi lapangan; kelompok SHA256 pada data belum menjamin foto ikan/sesi yang sama terpisah. Jangan memilih model/tingkat gangguan berdasarkan hasil test untuk lalu mengklaim uji yang sama sebagai konfirmasi independen.

In [ ]:
%pip -q install "timm>=1.0,<2" "scikit-learn==1.6.1" "lightgbm>=4,<5" "joblib>=1.3" "opencv-python-headless>=4.8" "pandas>=2" "matplotlib>=3.7" "seaborn>=0.13"

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys

drive.mount('/content/drive')
PROJECT = Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Folder proyek tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT'] = str(PROJECT)
sys.path.insert(0, str(PROJECT/'02_notebooks'))

from ffe_core import (paths, seed_all, ensure_dataset_local, load_split,
                      load_manifest, image_path, get_transform,
                      extract_feature_checkpoint, DeepOnlyNetFactory,
                      predict_logits, metric_row, CLASS_NAMES, test_group_integrity)
P = paths()
seed_all(42)
print('Workspace:', PROJECT)

## 1. Konfigurasi eksperimen

- **5 outer test folds**, masing-masing citra muncul tepat sekali di seluruh penggabungan out-of-fold.
- **7 kondisi**: foto asli; brightness gelap/terang, masing-masing dua tingkat; Gaussian blur dua tingkat.
- **3 classifier**: Deep-FCN, Deep-LR, Deep-LightGBM. Seluruh classifier memakai embedding dari backbone Swin-Tiny **yang sama pada fold yang sama**.
- **Tidak ada fitting, seleksi, atau tuning pada test set**. Semua parameter gangguan telah ditetapkan sebelum eksperimen dijalankan.
- `FULL_TEST = True` sebagai default: gunakan seluruh test set, bukan subset 120 sampel seperti notebook 07.
- `RESUME = True`: jika Colab terputus, hasil kondisi yang telah disimpan dapat digunakan kembali (dengan validasi ID sampel dan konfigurasi).

Perkiraan total input yang dievaluasi backbone: 4.390 × 7 = 30.730 citra. Jumlah prediksi classifier = 30.730 × 3.

In [ ]:
import json, hashlib, time, warnings
import numpy as np
import pandas as pd
import cv2, torch, timm, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FOLDS = [1,2,3,4,5]
MODELS = ['deep_fcn', 'deep_lr', 'deep_lgbm']
BATCH_SIZE = 24 if DEVICE.type=='cuda' else 8
FULL_TEST = True             # True untuk artikel; False hanya untuk uji coba awal
SMOKE_N_PER_FOLD = 24       # berlaku hanya jika FULL_TEST=False
RESUME = True
N_BOOT = 1000
BASE_SEED = 20261108
STRICT_BASELINE_PARITY = True
MAX_BASELINE_MISMATCH_RATE = 0.01  # periksa kompatibilitas preprocessing/model

CONDITIONS = [
 {'name':'original',        'kind':'original','severity':0,'factor':1.0},
 {'name':'dark_mild',       'kind':'dark','severity':1,'factor':0.80},
 {'name':'dark_strong',     'kind':'dark','severity':2,'factor':0.60},
 {'name':'bright_mild',     'kind':'bright','severity':1,'factor':1.20},
 {'name':'bright_strong',   'kind':'bright','severity':2,'factor':1.40},
 {'name':'blur_mild',       'kind':'blur','severity':1,'sigma':0.70},
 {'name':'blur_strong',     'kind':'blur','severity':2,'sigma':1.50},
]

RUN_DIR = 'robustness_jri' if FULL_TEST else 'robustness_jri_smoke'
OUT_METRICS = P['metrics']/RUN_DIR
OUT_FIGURES = P['figures']/RUN_DIR
PRED_DIR = OUT_METRICS/'predictions'
for q in [OUT_METRICS,OUT_FIGURES,PRED_DIR]:q.mkdir(parents=True,exist_ok=True)

PARAMS = {'conditions':CONDITIONS,'folds':FOLDS,'models':MODELS,
          'full_test':FULL_TEST,'smoke_n_per_fold':SMOKE_N_PER_FOLD,
          'seed':BASE_SEED,'label_order':CLASS_NAMES}
CONFIG_HASH=hashlib.sha256(json.dumps(PARAMS,sort_keys=True).encode()).hexdigest()[:16]
print('DEVICE:',DEVICE,'torch:',torch.__version__,'timm:',timm.__version__,
      'sklearn: 1.6.1 (pinned)','config:',CONFIG_HASH)
if not FULL_TEST:
 print('PERINGATAN: mode SMOKE. Hasil tidak boleh ditulis sebagai hasil final artikel.')

## 2. Pemeriksaan semua input

Notebook memerlukan artefak yang **sudah ada** dari notebook 00, 02, 03 dan 04. Tidak ada fitur handcrafted atau FusionNet yang digunakan dalam penelitian JRI versi sederhana ini.

In [ ]:
manifest = load_manifest()
assert len(manifest)==4390, f'Jumlah citra berubah: {len(manifest)}; audit ulang notebook 00.'
assert test_group_integrity(), 'Kebocoran hash/ID pada fold, periksa notebook 00.'

required=[]
for fold in FOLDS:
 required.extend([
  P['checkpoints']/f'swin_fold_{fold:02d}_best.pt',
  P['models']/f'deep_scaler_fold_{fold:02d}.joblib',
  P['models']/f'deep_lr_fold_{fold:02d}.joblib',
  P['models']/f'deep_lgbm_fold_{fold:02d}.joblib',
  P['models']/f'fold_{fold:02d}'/'deep_fcn.pt',
 ])
 for model in MODELS:
  required.append(P['metrics']/f'pred_{model}_fold_{fold:02d}.csv')
missing=[str(q) for q in required if not q.exists()]
assert not missing, 'Artefak belum lengkap, contoh file hilang:\n'+'\n'.join(missing[:15])
ensure_dataset_local()
print(f'OK: {len(required)} artefak/model baseline ditemukan; {len(manifest)} sampel dalam manifest.')
print('Output:',OUT_METRICS)

## 3. Fungsi transformasi gangguan dan inferensi

Citra asli dibaca dengan PIL dan menggunakan `get_transform(False)` yang **sama** dengan notebook 03. Gambar terganggu dibuat dari RGB asli, tetapi diberi transformasi yang telah dikunci sebelum prediksi. Gaussian blur menggunakan OpenCV. Keempat kondisi pencahayaan/blur tidak memodifikasi file pada dataset asli.

Catatan: peningkatan/penurunan brightness dilakukan terhadap nilai piksel RGB; ini gangguan sintetis, bukan simulasi sempurna kamera fisik.

In [ ]:
def perturb_image(rgb_pil, condition):
    """Input PIL RGB; output PIL RGB. Never writes source file."""
    kind=condition['kind']
    if kind=='original': return rgb_pil
    rgb=np.array(rgb_pil, dtype=np.uint8)
    if kind in ('dark','bright'):
        factor=float(condition['factor'])
        result=np.clip(np.rint(rgb.astype(np.float32)*factor),0,255).astype(np.uint8)
    elif kind=='blur':
        result=cv2.GaussianBlur(rgb,(0,0),sigmaX=float(condition['sigma']),
                                sigmaY=float(condition['sigma']))
    else: raise ValueError(f'Unknown kind: {kind}')
    return Image.fromarray(result,mode='RGB')

TRANSFORM=get_transform(False)

def extract_condition_embeddings(test, backbone, condition):
    """Exactly one Swin forward per image per condition; batch and order deterministic."""
    feats=[]
    backbone.eval()
    for start in range(0,len(test),BATCH_SIZE):
        chunk=test.iloc[start:start+BATCH_SIZE]
        tensors=[]
        for row in chunk.itertuples(index=False):
            with Image.open(image_path(row.rel_path)) as img:
                clean=img.convert('RGB')
                altered=perturb_image(clean,condition)
                tensors.append(TRANSFORM(altered))
        batch=torch.stack(tensors,dim=0).to(DEVICE)
        with torch.inference_mode():
            with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
                embeddings=backbone(batch)
        feats.append(embeddings.float().cpu().numpy())
        del batch,embeddings,tensors
    X=np.vstack(feats).astype(np.float32)
    assert X.shape==(len(test),768),f'Unexpected embedding shape {X.shape}'
    assert np.isfinite(X).all(),'Nonfinite deep embeddings'
    return X


def load_classifiers(fold):
    scaler=joblib.load(P['models']/f'deep_scaler_fold_{fold:02d}.joblib')
    checkpoint=torch.load(P['models']/f'fold_{fold:02d}'/'deep_fcn.pt',
                          map_location='cpu',weights_only=False)
    net=DeepOnlyNetFactory.create().to(DEVICE)
    net.load_state_dict(checkpoint['state_dict'])
    net.eval()
    lr=joblib.load(P['models']/f'deep_lr_fold_{fold:02d}.joblib')
    lgbm=joblib.load(P['models']/f'deep_lgbm_fold_{fold:02d}.joblib')
    assert np.array_equal(lr.classes_,[0,1,2])
    assert np.array_equal(lgbm.classes_,[0,1,2])
    return net,lr,lgbm,scaler


def make_predictions(X,test,fold,condition,models):
    net,lr,lgbm,scaler=models
    X_scaled=scaler.transform(X).astype(np.float32)
    fc_logits=predict_logits(net,X_scaled,bs=512,device=DEVICE)
    fc_prob=torch.softmax(torch.from_numpy(fc_logits),dim=1).numpy()
    probs={'deep_fcn':fc_prob,
           'deep_lr':lr.predict_proba(X),
           'deep_lgbm':lgbm.predict_proba(X)}
    out=test[['sample_id','species','freshness','label']].rename(columns={'label':'y_true'}).copy()
    out.insert(0,'fold',fold)
    out.insert(1,'condition',condition['name'])
    out['config_hash']=CONFIG_HASH
    for model,pr in probs.items():
        assert pr.shape==(len(out),3) and np.isfinite(pr).all()
        assert np.allclose(pr.sum(axis=1),1.,rtol=1e-3,atol=1e-3)
        out[f'{model}_pred']=np.argmax(pr,axis=1).astype(int)
        for k in range(3):out[f'{model}_p{k}']=pr[:,k]
    return out


def assert_prediction_parity(pred_df,fold):
    """Catch mismatched Swin input preprocessing, embedding, scaler or models."""
    report=[]
    for model in MODELS:
        prior=pd.read_csv(P['metrics']/f'pred_{model}_fold_{fold:02d}.csv',
                          dtype={'sample_id':str})
        prior=prior.set_index('sample_id')
        ids=pred_df.sample_id.astype(str).tolist()
        assert set(ids).issubset(set(prior.index)),f'Unexpected sample IDs: {model}'
        old=prior.loc[ids,'y_pred'].to_numpy(dtype=int)
        new=pred_df[f'{model}_pred'].to_numpy(dtype=int)
        mismatch=int(np.sum(old!=new))
        rate=mismatch/len(new)
        report.append({'fold':fold,'model':model,'n':len(new),
                       'mismatch_n':mismatch,'mismatch_rate':rate})
        if rate>MAX_BASELINE_MISMATCH_RATE and STRICT_BASELINE_PARITY:
            raise RuntimeError(
                f'Baseline {model} fold={fold}: {mismatch}/{len(new)} prediksi berbeda '
                f'dengan notebook sebelumnya. Periksa timm/PIL/transform/scaler sebelum lanjut.')
    return report

## 4. Eksekusi 5 fold × 7 kondisi

Satu baris keluaran berarti prediksi **tiga classifier pada satu citra di satu kondisi**. Hasil per fold-condition disimpan otomatis dan dapat dilanjutkan jika runtime terputus. Simpan file di Drive sebelum keluar dari runtime.

In [ ]:
all_predictions=[]
parity_reports=[]
t0=time.time()
for fold in FOLDS:
    print(f'\n=== Fold {fold}/{len(FOLDS)} ===')
    test=load_split(fold,'test').reset_index(drop=True)
    if not FULL_TEST:
        test=test.sample(n=min(SMOKE_N_PER_FOLD,len(test)),random_state=BASE_SEED+fold).reset_index(drop=True)
    fold_dir=PRED_DIR/f'fold_{fold:02d}';fold_dir.mkdir(parents=True,exist_ok=True)
    needed=[]
    for condition in CONDITIONS:
        fn=fold_dir/f'{condition["name"]}.csv'
        if not (RESUME and fn.exists()):needed.append(condition)
    backbone=None;models=None
    try:
        if needed:
            backbone=extract_feature_checkpoint(P['checkpoints']/f'swin_fold_{fold:02d}_best.pt',DEVICE)
            models=load_classifiers(fold)
        for condition in CONDITIONS:
            fn=fold_dir/f'{condition["name"]}.csv'
            if RESUME and fn.exists():
                result=pd.read_csv(fn,dtype={'sample_id':str,'config_hash':str})
                if not (len(result)==len(test) and
                        result.sample_id.tolist()==test.sample_id.astype(str).tolist() and
                        (result.config_hash==CONFIG_HASH).all()):
                    raise RuntimeError(f'Invalid cached file: {fn}. Delete it to recompute.')
                print(f'  [cache] {condition["name"]}: {len(result)}')
            else:
                X=extract_condition_embeddings(test,backbone,condition)
                result=make_predictions(X,test,fold,condition,models)
                if condition['name']=='original':
                    report=assert_prediction_parity(result,fold)
                    parity_reports.extend(report)
                    print('  Original parity mismatches:',{x['model']:x['mismatch_n'] for x in report})
                result.to_csv(fn,index=False)
                print(f'  [saved] {condition["name"]}: {len(result)}')
            if condition['name']=='original' and not any(x['fold']==fold for x in parity_reports):
                parity_reports.extend(assert_prediction_parity(result,fold))
            all_predictions.append(result)
    finally:
        if backbone is not None:del backbone
        if models is not None:del models
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    print('Elapsed minutes:',round((time.time()-t0)/60,1))

preds=pd.concat(all_predictions,ignore_index=True)
assert preds.groupby('condition').size().nunique()==1
assert preds.groupby('condition').size().iloc[0]==(4390 if FULL_TEST else 5*SMOKE_N_PER_FOLD)
assert not preds.duplicated(['sample_id','condition']).any()
assert set(preds.condition)==set(x['name'] for x in CONDITIONS)
parity_df=pd.DataFrame(parity_reports)
parity_df.to_csv(OUT_METRICS/'baseline_parity.csv',index=False)
print(f'Completed: {len(preds):,} rows, {len(parity_df)} parity checks, {time.time()-t0:.1f} seconds')
display(parity_df)

## 5. Metrik performa: keseluruhan, per fold, per spesies, dan per kelas

**Jangan menilai robustness hanya dari akurasi.** Kita juga mengukur Macro F1 dan recall kelas `Not Fresh`, serta berapa kali `Not Fresh` salah diklasifikasikan sebagai `Highly Fresh`.

In [ ]:
def metrics_df(df,model):
    y=df.y_true.to_numpy(dtype=int)
    p=df[f'{model}_pred'].to_numpy(dtype=int)
    ans=metric_row(y,p)
    cm=confusion_matrix(y,p,labels=[0,1,2])
    ans.update({'n':len(df),
                'not_fresh_recall': float(cm[2,2]/max(1,cm[2,:].sum())),
                'not_fresh_to_highly_fresh_n':int(cm[2,0]),
                'not_fresh_to_highly_fresh_rate':float(cm[2,0]/max(1,cm[2,:].sum()))})
    return ans

pooled=[];per_fold=[];per_species=[];per_class=[]
for condition in CONDITIONS:
    sub=preds[preds.condition==condition['name']]
    for model in MODELS:
        pooled.append({'condition':condition['name'],'kind':condition['kind'],
                       'severity':condition['severity'],'model':model,
                       **metrics_df(sub,model)})
        for fold,g in sub.groupby('fold',sort=True):
            per_fold.append({'fold':int(fold),'condition':condition['name'],'model':model,
                             **metrics_df(g,model)})
        for species,g in sub.groupby('species',sort=True):
            per_species.append({'condition':condition['name'],'model':model,
                                'species':species,**metrics_df(g,model)})
        y=sub.y_true.to_numpy(dtype=int)
        p=sub[f'{model}_pred'].to_numpy(dtype=int)
        for label_i,label_name in enumerate(CLASS_NAMES):
            mask=y==label_i
            recall=float(np.mean(p[mask]==label_i)) if np.any(mask) else np.nan
            per_class.append({'condition':condition['name'],'model':model,
                              'class':label_name,'n':int(mask.sum()),'recall':recall})

pool=pd.DataFrame(pooled);fold_metrics=pd.DataFrame(per_fold)
species_metrics=pd.DataFrame(per_species);class_metrics=pd.DataFrame(per_class)
for name,frame in [('pooled_metrics.csv',pool),('fold_metrics.csv',fold_metrics),
                   ('per_species.csv',species_metrics),('per_class_recall.csv',class_metrics)]:
    frame.to_csv(OUT_METRICS/name,index=False)
display(pool[['condition','model','n','accuracy','macro_f1','mcc','not_fresh_recall']].round(4))

## 6. Paired robustness dan interval bootstrap 95%

Pengujian berpasangan dilakukan pada **ID sampel yang sama**. Ukuran utama adalah perubahan Macro F1 dari kondisi normal ke kondisi terganggu. Jika penurunan suatu model lebih kecil, model tersebut **lebih stabil secara deskriptif** pada gangguan itu. Interval bootstrap yang mencakup nol berarti belum cukup bukti untuk menyatakan perbedaan yang konsisten.

**Keterbatasan statistik:** bootstrap dilakukan pada citra karena dataset belum menyediakan ID ikan atau sesi. Bila beberapa gambar berasal dari ikan/sesi yang sama, interval bisa terlalu sempit. Oleh karena itu gunakan kata *image-level exploratory confidence intervals* dalam artikel.

In [ ]:
def macro_f1_fast(y,p,n_classes=3):
    cm=np.bincount(y*n_classes+p,minlength=n_classes*n_classes).reshape(n_classes,n_classes)
    tp=np.diag(cm)
    denom=cm.sum(axis=1)+cm.sum(axis=0)
    return float(np.mean(np.divide(2*tp,denom,out=np.zeros(n_classes,float),where=denom!=0)))


def paired_bootstrap_ci(y,p_a,p_b,n_boot=N_BOOT,seed=42):
    """Return F1(A)-F1(B) and paired image bootstrap percentile CI."""
    y=np.asarray(y,dtype=int);a=np.asarray(p_a,dtype=int);b=np.asarray(p_b,dtype=int)
    assert len(y)==len(a)==len(b)
    point=macro_f1_fast(y,a)-macro_f1_fast(y,b)
    rng=np.random.default_rng(seed)
    values=np.empty(n_boot,dtype=float)
    for i in range(n_boot):
        ix=rng.integers(0,len(y),size=len(y))
        values[i]=macro_f1_fast(y[ix],a[ix])-macro_f1_fast(y[ix],b[ix])
    return point, np.quantile(values,[.025,.975])


# Normal dan perturbasi selalu diselaraskan lewat sample_id; bukan bergantung pada urutan CSV.
normal=preds[preds.condition=='original'].set_index('sample_id')
compare=[];relative=[]
for condition in CONDITIONS[1:]:
    pert=preds[preds.condition==condition['name']].set_index('sample_id')
    assert set(normal.index)==set(pert.index)
    pert=pert.loc[normal.index]
    y=normal.y_true.to_numpy(dtype=int)
    assert np.array_equal(y,pert.y_true.to_numpy(dtype=int))
    for model in MODELS:
        original_p=normal[f'{model}_pred'].to_numpy(dtype=int)
        corrupted_p=pert[f'{model}_pred'].to_numpy(dtype=int)
        # F1(corrupted) - F1(normal): nilai negatif berarti degradasi.
        point,ci=paired_bootstrap_ci(y,corrupted_p,original_p,
                    seed=BASE_SEED+sum(map(ord,condition['name']+model)))
        compare.append({'condition':condition['name'],'kind':condition['kind'],
                        'severity':condition['severity'],'model':model,
                        'macro_f1_change':point,'ci95_low':ci[0],'ci95_high':ci[1],
                        'change_per_100_points':100*point,
                        'n':len(y)})
        if model!='deep_fcn':
            # Model X vs Deep-FCN *pada kondisi sama*; nilai positif = unggul pada kondisi itu.
            delta,ci2=paired_bootstrap_ci(y,corrupted_p,
                      pert['deep_fcn_pred'].to_numpy(dtype=int),
                      seed=BASE_SEED+10000+sum(map(ord,condition['name']+model)))
            relative.append({'condition':condition['name'],'model':model,
                             'versus':'deep_fcn','macro_f1_diff_on_corrupted':delta,
                             'ci95_low':ci2[0],'ci95_high':ci2[1],'n':len(y)})
change=pd.DataFrame(compare);vs_fcn=pd.DataFrame(relative)
change.to_csv(OUT_METRICS/'paired_f1_change_bootstrap.csv',index=False)
vs_fcn.to_csv(OUT_METRICS/'model_vs_fcn_on_corrupted_bootstrap.csv',index=False)

# Uji pertanyaan ROBUSTNESS yang paling relevan: selisih tingkat penurunan.
# Positif = model pembanding mengalami penurunan Macro F1 yang lebih kecil dari FCN.
def paired_drop_advantage_ci(y, model_orig, model_pert, fcn_orig, fcn_pert,
                             n_boot=N_BOOT, seed=42):
    def diff(y,mo,mp,fo,fp):
        return ((macro_f1_fast(y,mp)-macro_f1_fast(y,mo)) -
                (macro_f1_fast(y,fp)-macro_f1_fast(y,fo)))
    point=diff(y,model_orig,model_pert,fcn_orig,fcn_pert)
    rng=np.random.default_rng(seed)
    samples=np.empty(n_boot,float)
    for i in range(n_boot):
        ix=rng.integers(0,len(y),len(y))
        samples[i]=diff(y[ix],model_orig[ix],model_pert[ix],fcn_orig[ix],fcn_pert[ix])
    return point,np.quantile(samples,[.025,.975])

robustness_adv=[]
for condition in CONDITIONS[1:]:
    pert=preds[preds.condition==condition['name']].set_index('sample_id').loc[normal.index]
    y=normal.y_true.to_numpy(dtype=int)
    for model in ['deep_lr','deep_lgbm']:
        point,ci=paired_drop_advantage_ci(
            y,normal[f'{model}_pred'].to_numpy(dtype=int),
            pert[f'{model}_pred'].to_numpy(dtype=int),
            normal['deep_fcn_pred'].to_numpy(dtype=int),
            pert['deep_fcn_pred'].to_numpy(dtype=int),
            seed=BASE_SEED+20000+sum(map(ord,condition['name']+model)))
        robustness_adv.append({'condition':condition['name'],'model':model,
            'reference_model':'deep_fcn','advantage_in_f1_change':point,
            'ci95_low':ci[0],'ci95_high':ci[1],'n':len(y)})
robustness_adv=pd.DataFrame(robustness_adv)
robustness_adv.to_csv(OUT_METRICS/'robustness_advantage_vs_fcn_bootstrap.csv',index=False)
print('Paired change: F1(corrupted) - F1(original)')
display(change.round(4))
print('Robustness advantage versus Deep-FCN: positive means smaller performance drop')
display(robustness_adv.round(4))

## 7. Grafik siap dipakai pada artikel

Grafik menyajikan seluruh data out-of-fold (bukan subset 120 gambar), memisahkan tiga gangguan dan dua tingkat intensitas. File resolusi 200 dpi disimpan sebagai PNG untuk draft jurnal.

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
MODEL_LABEL={'deep_fcn':'Swin + FCN','deep_lr':'Swin + LR','deep_lgbm':'Swin + LightGBM'}

# 1. Grafik perubahan Macro F1 terhadap kondisi asli
fig,ax=plt.subplots(figsize=(10,5))
order=[x['name'] for x in CONDITIONS]
for model in MODELS:
    series=pool[pool.model==model].set_index('condition').loc[order]
    ax.plot(range(len(order)),series.macro_f1,marker='o',linewidth=2,label=MODEL_LABEL[model])
ax.set_xticks(range(len(order)),[x.replace('_','\n') for x in order])
ax.set_ylabel('Macro F1 (pooled out-of-fold)')
ax.set_title('Performance under controlled image degradation')
ax.grid(axis='y',alpha=.25);ax.legend(loc='best');fig.tight_layout()
fig.savefig(OUT_FIGURES/'macro_f1_by_condition.png',dpi=200,bbox_inches='tight')
plt.show()

# 2. Delta terhadap kondisi normal: lebih dekat nol = penurunan lebih kecil
fig,ax=plt.subplots(figsize=(10,4.5))
for model in MODELS:
    df=change[change.model==model].set_index('condition').loc[order[1:]]
    ax.plot(range(len(df)),df.macro_f1_change,marker='o',linewidth=2,label=MODEL_LABEL[model])
ax.axhline(0,color='black',linewidth=1,linestyle='--')
ax.set_xticks(range(len(order)-1),[x.replace('_','\n') for x in order[1:]])
ax.set_ylabel('Δ Macro F1 (corrupted − original)')
ax.set_title('Paired degradation from original images')
ax.grid(axis='y',alpha=.25);ax.legend(loc='best');fig.tight_layout()
fig.savefig(OUT_FIGURES/'paired_macro_f1_drop.png',dpi=200,bbox_inches='tight')
plt.show()

# 3. Kesalahan penting: Not Fresh diprediksi Highly Fresh
pivot=pool.pivot(index='condition',columns='model',values='not_fresh_to_highly_fresh_rate').loc[order]
fig,ax=plt.subplots(figsize=(8.5,5))
sns.heatmap(pivot.rename(columns=MODEL_LABEL),annot=True,fmt='.3f',cmap='Reds',ax=ax)
ax.set_title('Not Fresh misclassified as Highly Fresh: within-class rate')
ax.set_ylabel('Condition');ax.set_xlabel('Classifier');fig.tight_layout()
fig.savefig(OUT_FIGURES/'critical_error_heatmap.png',dpi=200,bbox_inches='tight')
plt.show()

## 8. Ringkasan otomatis untuk penulisan artikel dan rekam jejak eksperimen

Menyimpan konfigurasi, ringkasan, serta keputusan yang aman untuk pembahasan. **Tidak** mengklaim signifikansi hanya berdasarkan model dengan angka tertinggi.

In [ ]:
config_payload={**PARAMS,'config_hash':CONFIG_HASH,'device':str(DEVICE),
                'torch_version':torch.__version__,'timm_version':timm.__version__,
                'n_predictions':len(preds),'n_unique_images':int(preds.sample_id.nunique()),
                'statistical_note':'Paired image-level bootstrap; fish/session independence unverified',
                'research_note':'Synthetic perturbation, not real-world acquisition validation'}
(OUT_METRICS/'experiment_config.json').write_text(json.dumps(config_payload,indent=2),encoding='utf-8')

normal_metrics=pool[pool.condition=='original'].set_index('model')
rows=['# Ringkasan eksperimen robustness — JRI',
      '',f'- Status: {"FULL" if FULL_TEST else "SMOKE / NOT FOR PAPER"}',
      f'- N gambar unik: {preds.sample_id.nunique()}',
      f'- Fold: {FOLDS}',
      '- Kondisi: '+', '.join(x['name'] for x in CONDITIONS),
      '', '## Baseline asli (pooled OOF)',
      '', '| Model | Accuracy | Macro F1 |', '|---|---:|---:|']
for m in MODELS:
    x=normal_metrics.loc[m]
    rows.append(f'| {MODEL_LABEL[m]} | {x.accuracy:.4f} | {x.macro_f1:.4f} |')
rows.extend(['','## Perubahan Macro F1 (kondisi − asli)','',
             '| Condition | Model | Δ Macro F1 | 95% image-bootstrap CI |',
             '|---|---|---:|---|'])
for r in change.itertuples(index=False):
    rows.append(f'| {r.condition} | {MODEL_LABEL[r.model]} | {r.macro_f1_change:+.4f} | [{r.ci95_low:+.4f}, {r.ci95_high:+.4f}] |')
rows.extend(['','## Batasan wajib dilaporkan','',
             '1. Citra terganggu dihasilkan secara sintetis, bukan pengambilan baru di pasar.',
             '2. SHA256-grouped CV tidak menguji independensi individu ikan atau sesi kamera.',
             '3. Bootstrap berpasangan pada level citra berisiko mengabaikan korelasi antarrekaman.',
             '4. Model dan perturbasi memakai sampel test yang sama; hasil untuk eksplorasi performa, bukan bukti eksternal.',
             '5. Perbedaan antar-model tidak otomatis signifikan. Gunakan interval dan nilai metrik konkret.',
             '', '## File keluaran',
             '- pooled_metrics.csv; fold_metrics.csv; per_species.csv; per_class_recall.csv',
             '- paired_f1_change_bootstrap.csv; model_vs_fcn_on_corrupted_bootstrap.csv',
             '- robustness_advantage_vs_fcn_bootstrap.csv: selisih penurunan vs Deep-FCN',
             '- baseline_parity.csv; experiment_config.json; prediction CSV per fold dan kondisi',
             '- figures: macro_f1_by_condition.png; paired_macro_f1_drop.png; critical_error_heatmap.png'])
(OUT_METRICS/'JRI_findings_readme.md').write_text('\n'.join(rows),encoding='utf-8')

print('\nSelesai. Semua artefak tersedia di:')
print(OUT_METRICS)
print(OUT_FIGURES)
print('PERINGATAN: verifikasi hasil, jangan menyimpulkan robustness lebih baik hanya dari satu nilai terbaik.')